In [27]:
import tiktoken
import re
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn


In [28]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}

SMOKE_TEST_CONFIG = {
    "vocab_size": 50257,
    "context_length": 4,
    "emb_dim": 16,
    "n_heads": 4,
    "n_layers": 2,
    "drop_rate": 0.0,
    "qkv_bias": False,
}


In [29]:
class SimpleTokenizer:
    def __init__(self, vocab):
        self.str2int = vocab
        self.int2str = {i : s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r"([,.:;?_!()'\"]|--|\s)", text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        return [self.str2int[token] if token in self.str2int else self.str2int["<|unk|>"]  
                for token in preprocessed]
    def decode(self, ids):
        text = " ".join([self.int2str[i] for i in ids])
        text = re.sub(r"\s+([,.:;?_!()'\"]|--)", r'\1', text)
        return text


模块名：dataset
输入：text
输出：(input_ids, output_ids)的tensor
内部状态：text-ids, input_ids, output_ids
核心步骤：先str2ids得到input，再根据滑窗取target
边界情况：文本长度小于数据集长度


In [30]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(text)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1:i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]


In [31]:
#data loader负责取dataset
def create_dataloader(
        text, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(text, tokenizer, max_length, stride)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataloader
text = "hello world,hello world, this is a small test text for the dataloader"


In [32]:
class GPTInputEmbedding(nn.Module):
    def __init__(self, vocab, emb_dim, context_length):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab, emb_dim)
        self.position_embedding = nn.Embedding(context_length, emb_dim)

    def forward(self, token_ids):
        seq_len = token_ids.shape[1]
        positions = torch.arange(seq_len, device=token_ids.device)
        token_embeds = self.token_embedding(token_ids)
        pos_embeds = self.position_embedding(positions)
        input_embeds = token_embeds + pos_embeds
        return input_embeds


In [33]:
#实现attention
class CausalAttention(nn.Module):
    def __init__(self, emb_dim, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_q=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.W_k=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.W_v=nn.Linear(emb_dim, d_out, bias=qkv_bias)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length,context_length, dtype=torch.bool),
            diagonal=1
        )
        )
        self.dropout=nn.Dropout(dropout)
    
    def forward(self, emb_vec, return_attn=False):
        b, seq_len, emb_dim = emb_vec.shape
        queries = self.W_q(emb_vec) #[B, T, D]
        keys = self.W_k(emb_vec)  
        values = self.W_v(emb_vec)
        d_k = keys.shape[-1]
        attention_scores = queries @ keys.transpose(1,2) / (d_k**0.5) #[B, T, T]
        masked_att = attention_scores.masked_fill(self.mask[:seq_len, :seq_len], -torch.inf)
        attention_weights = torch.softmax(masked_att, dim=-1)
        attention_weights = self.dropout(attention_weights)
        context_vec = attention_weights @ values
        if return_attn:
            return context_vec, attention_weights
        return context_vec


In [34]:
#简易版multihead实现
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, emb_dim, context_length, d_out, dropout, num_head, qkv_bias=False):
        super().__init__()
        self.heads=nn.ModuleList(CausalAttention(
            emb_dim=emb_dim,
            d_out=d_out,
            context_length=context_length,
            dropout=dropout,
            qkv_bias=qkv_bias
        )
        for i in range(0, num_head)
        )
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


In [35]:
'''multihead完全版实现
初始化参数：d_in, d_out, context_length, dropout, num_heads, qkv_bias
forward 输入：x，形状 [batch_size, num_tokens, d_in]
forward 输出：context_vec，形状 [batch_size, num_tokens, d_out]
输出：合并版的上下文向量
核心步骤：
0.初始化得到最大qkv矩阵
1.根据输出维度和多头数量得到每一个注意力头的输出维度c
1.a.分为多头
2.计算相应的注意力分数以及softmax化和dropout
3.通过view方法把注意力转换为多头
3.把单头得到的上下文向量通过torch.view结合起来，contigus()方法进行reshape
4.通过线性投影层，输出上下文向量
'''
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, context_length, dropout, d_out, num_heads,qkv_bias=False) -> None:
        super().__init__()
        self.W_q=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_k=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_v=nn.Linear(d_in, d_out, bias=qkv_bias)
        self.d_out= d_out #保存一下下面forward使用
        self.num_heads=num_heads#保存一下下面forward使用
        self.dropout=nn.Dropout(dropout)
        self.out_proje=nn.Linear(d_out,d_out)
        self.register_buffer(
            'mask',
            torch.triu(
                torch.ones(context_length,context_length, dtype=torch.bool),#注意写上bool值才能让这个矩阵变成布尔矩阵 
                diagonal=1)
        )
        assert d_out % num_heads == 0, "d_out 必须被 num_heads 整除"
        self.head_dim=d_out // num_heads #view期待init,/返回float，所以要用//
    def forward(self, x):
        batch_size, seq_length, d_in = x.shape #[B,T,I]
        d_out=self.d_out#d_out = self.W_q.shape[-1]错误，linear模块没有shape查询
        num_heads=self.num_heads
        queries=self.W_q(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)#[B,T,N,H]->[B,N,T,H]
        keys=self.W_k(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)
        values=self.W_v(x).view(batch_size, seq_length, num_heads, self.head_dim).transpose(1,2)
        d_k = keys.shape[-1]
        attention_scores = queries @ keys.transpose(2,3) / d_k**0.5 #[B,N,T,T]
        masked_attn=torch.masked_fill(attention_scores, self.mask[:seq_length, :seq_length], -torch.inf)
        attn_weight = torch.softmax(masked_attn, -1)
        attn_weight = self.dropout(attn_weight)
        s_context_vec = attn_weight @ values#[B, N, T, H]
        context_vec = s_context_vec.transpose(1,2).contiguous().view(batch_size, seq_length, d_out)#这里可以优化，上面已经保存self.d_out了，不用再重新赋值
        return self.out_proje(context_vec)


In [36]:
B = 2
T = 4
emb_dim = 16
d_out = 68
num_heads = 4
context_length = 4

x = torch.randn(B, T, emb_dim)

mha = MultiHeadAttention(
    d_in=emb_dim,
    d_out=d_out,
    context_length=context_length,
    dropout=0.0,
    num_heads=num_heads
)

out = mha(x)
print(out.shape)
print(out)


torch.Size([2, 4, 68])
tensor([[[-2.0770e-01, -4.6965e-01,  8.3448e-02, -1.4606e-01, -1.5862e-01,
          -5.8791e-02,  2.0730e-02,  7.0633e-02, -1.1660e-01, -6.6113e-01,
           4.7097e-01, -6.6103e-01,  5.3570e-01, -2.7976e-01,  4.7061e-02,
          -4.6961e-01, -3.7567e-01,  3.3062e-01, -2.7483e-01,  6.9293e-01,
           2.2407e-01, -4.4126e-01,  4.2815e-01, -2.5969e-01,  1.3499e-01,
          -3.7941e-02,  7.0504e-01,  2.4464e-01, -1.5272e-01, -4.4347e-03,
          -4.9102e-01, -3.3955e-01, -1.0217e+00,  1.3955e-01, -1.8100e-01,
           2.2701e-01, -1.4422e-01, -1.6421e-01, -1.5212e-01,  1.0635e-01,
          -1.9243e-01,  1.3690e-03,  1.2933e-01, -6.2369e-01, -7.3997e-01,
           3.3765e-01, -4.7172e-01, -5.9114e-01,  2.9032e-01,  3.2453e-01,
           6.7952e-01, -2.8080e-01,  5.2617e-02, -3.9393e-01,  1.0417e-01,
          -5.2108e-02,  4.3930e-01,  8.4140e-04,  1.8361e-01,  2.6775e-01,
           7.2753e-01,  1.7294e-01,  6.1753e-02, -2.5434e-01, -4.1823e-01,
  

In [37]:
'''归一化和GELU激活函数、前馈神经网络实现
输入：x [batch_size, seq_length, d_out]
输出：处理后的x，维度不变，但-1维均值为0，方差为1（为了让之后梯度下降更快收敛、训练更稳定）
核心目标：
1.归一化函数，用mean和var实现，先将每个 token 的最后一维归一化为均值约 0、方差约 1，
再通过可学习的 scale 和 bias 做缩放和平移。 #最好定为class，因为它有可训练参数
2.GELU激活函数，输入输出shape不变
3.FNN内交替调用神经层和GELU，中间隐藏层d_out增加
'''
class LayerNorm(nn.Module):
    def __init__(self, d_in):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_in)) #[d]
        self.bias = nn.Parameter(torch.zeros(d_in))
        self.eps = 1e-5
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)#[b,s,1]
        var = x.var(dim=-1, keepdim=True, unbiased=False) #用总体var，unbiased为false:确保d小的时候var为1
        norm_x = (x-mean) / (var+self.eps)**0.5 #[b,s,d],可用torch.sqrt(...)替换        
        return norm_x*self.scale + self.bias #每个vec的d维度对应乘上self.scale d的相应索引值再加上self.bias的相应d索引值
class GELU(nn.Module): #也写成class,方便后面sequential使用
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi, device=x.device)) * (x + 0.044715 * x**3)
        ))
class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_mult=4):
        super().__init__()
        self.fnn = nn.Sequential(
            nn.Linear(emb_dim, hidden_mult * emb_dim),
            GELU(),
            nn.Linear(hidden_mult*emb_dim, emb_dim)
        )
    def forward(self, x):
        return self.fnn(x) #[b,s,d]


In [38]:
b = 2
s= 4
e = 3
x = torch.randn(b,s,e)
norm = LayerNorm(e)
fnn = FeedForward(e)
print(fnn(x).shape)
print(norm(x).shape)


torch.Size([2, 4, 3])
torch.Size([2, 4, 3])


In [39]:
#实现transformer、残差连接。输入：emb_dim,drop,tensor x,输出修改后的x。两个归一化层，两次drop，Pre-LN
class Transformer(nn.Module):
    def __init__(self, emb_dim, dropout, num_heads, context_length, qkv_bias=False):
        super().__init__()
        self.mha = MultiHeadAttention(
            d_in=emb_dim,
            context_length=context_length,
            d_out=emb_dim,
            dropout=dropout,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
        )
        self.ffn=FeedForward(emb_dim)
        self.norm1=LayerNorm(emb_dim)
        self.norm2=LayerNorm(emb_dim)
        self.dropout=nn.Dropout(dropout)

    def forward(self, x):
        #attn
        shortcut = x
        x = self.norm1(x)
        x = self.mha(x)
        x = self.dropout(x)
        x += shortcut
        #ff
        shortcut = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = self.dropout(x)
        x += shortcut
        return x


In [40]:
#GPT实现
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.input_embedding = GPTInputEmbedding(
            vocab=cfg["vocab_size"],
            emb_dim=cfg["emb_dim"],
            context_length=cfg["context_length"]
        )

        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[Transformer(
                emb_dim=cfg["emb_dim"],
                dropout=cfg["drop_rate"],
                num_heads=cfg["n_heads"],
                context_length=cfg["context_length"],
                qkv_bias=cfg["qkv_bias"],
            ) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"],
            cfg["vocab_size"],
            bias=False
        )

    def forward(self, in_idx):
        x = self.input_embedding(in_idx)   # [B, T, emb_dim]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)          # [B, T, vocab_size]
        return logits


In [41]:
#用小配置把上面的模块串起来，确认 GPT 前向流程能跑通
'''
model = GPTModel(SMOKE_TEST_CONFIG)
input_ids = torch.randint(
    low=0,
    high=SMOKE_TEST_CONFIG["vocab_size"],
    size=(2, SMOKE_TEST_CONFIG["context_length"]),
)
logits = model(input_ids)
print(logits.shape)
assert logits.shape == (2, SMOKE_TEST_CONFIG["context_length"], SMOKE_TEST_CONFIG["vocab_size"])
'''

'\nmodel = GPTModel(SMOKE_TEST_CONFIG)\ninput_ids = torch.randint(\n    low=0,\n    high=SMOKE_TEST_CONFIG["vocab_size"],\n    size=(2, SMOKE_TEST_CONFIG["context_length"]),\n)\nlogits = model(input_ids)\nprint(logits.shape)\nassert logits.shape == (2, SMOKE_TEST_CONFIG["context_length"], SMOKE_TEST_CONFIG["vocab_size"])\n'

In [42]:
'''生成逻辑实现：
第一版贪心解码：
输入：文本，tokenizer，max_new_tokens，top_k,temperature
外层wrapper：负责文本到token_ids，token ids到文本
核心generation处理张量循环
'''
def generation(token_ids, max_new_tokens, model, context_size, temperature, top_k=None,eos_id=None):
    device = next(model.parameters()).device
    token_ids = token_ids.to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = token_ids[: , -context_size:] #[B, T]
            logits = model(context) #[B, T, vocab]
                #对最后一个生成出来的logits进行排序取样
            last_logits = logits[:, -1, :] #[B,V]
            if top_k is not None:
                #取最小的logit与原tensor比较，在归一化前把top_k外的值排除
                p_toplogits, _ = torch.topk(last_logits, top_k)
                min_logits = p_toplogits[:, -1].unsqueeze(1)
                last_logits = torch.where(last_logits < min_logits, -torch.inf, last_logits)
            if temperature == 0:
                idx_next = torch.argmax(last_logits, dim=1, keepdim=True) #[B,1]
            elif temperature > 0:
                prob = torch.softmax(last_logits / temperature, dim=-1)
                idx_next = torch.multinomial(prob, 1)
            if eos_id is not None and (idx_next == eos_id).all().item():
                break
            token_ids = torch.cat((token_ids, idx_next), dim=1)
    return token_ids
 

In [43]:
'''
cfg = GPT_CONFIG_124M
token_ids = torch.randint(low=0, high=cfg["vocab_size"], size=(2, 4))
model = GPTModel(cfg)
model.eval()
y1 = generation(token_ids.clone(), 10, model, 4, 1,2)
y2 = generation(token_ids.clone(), 10, model, 4, 0)
print(y1)
print(y2)
print(torch.equal(y1, y2))''' 

'\ncfg = GPT_CONFIG_124M\ntoken_ids = torch.randint(low=0, high=cfg["vocab_size"], size=(2, 4))\nmodel = GPTModel(cfg)\nmodel.eval()\ny1 = generation(token_ids.clone(), 10, model, 4, 1,2)\ny2 = generation(token_ids.clone(), 10, model, 4, 0)\nprint(y1)\nprint(y2)\nprint(torch.equal(y1, y2))'

In [44]:
def generate_text(text, tokenizer, model, cfg,temperature=0,top_k=None):
    tokenizer=tokenizer
    encoded_tensor = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
    cfg = cfg
    model = model
    out_tensor = generation(encoded_tensor, 10, model,cfg["context_length"],temperature,top_k)
    return tokenizer.decode(out_tensor.squeeze(0).tolist())



In [45]:
#初始化模型试验
cfg=GPT_CONFIG_124M
tokenizer=tiktoken.get_encoding("gpt2")
tem=1
top_k=5
model=GPTModel(cfg)
text = "Hello, world"
new_text=generate_text(text,tokenizer,model,cfg,tem,top_k)
print(new_text)

Hello, worldocadoIslamic lasers crouodikick Lies DeV Needmajor


In [46]:
#loss计算实现
def cal_loss_batch(input_batch, target_batch, model):
    device = next(model.parameters()).device
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    #把logit和target变成交叉熵能处理的方式，[B,T,V]->[N,V], [B,T]->[N]
    logits = torch.flatten(logits,0,1)
    label = torch.flatten(target_batch)
    loss = nn.functional.cross_entropy(logits, label)
    return loss

In [47]:
input_batch = torch.randint(0,50257,(2,5),dtype=torch.long)
target_batch = torch.randint(0,50257,(2,5),dtype=torch.long)
loss = cal_loss_batch(input_batch, target_batch, model)
print(loss.shape)
print(loss.item())
print(loss)

torch.Size([])
10.98646354675293
tensor(10.9865, grad_fn=<NllLossBackward0>)


In [48]:
def cal_loss_loader(dataloader, model,num_batches=None):
    total_loss = 0
    assert len(dataloader) > 0 , "nothing in dataloader"
    if num_batches is None:
        num_batches = len(dataloader)
    else:
        num_batches = min(num_batches, len(dataloader))
    for b, (input_batch,target_batch) in enumerate(dataloader):
        if b >= num_batches:
            break
        loss = cal_loss_batch(input_batch, target_batch, model)
        total_loss += loss.item() #减少计算图
    avg_loss = total_loss / num_batches
    return avg_loss

In [49]:
def generate_text_sample(text, model, tokenizer):
    model.eval()
    context_size = model.input_embedding.position_embedding.weight.shape[0]
    encoded_tensor = torch.tensor(tokenizer.encode(text)).unsqueeze(0).to(next(model.parameters()).device)
    with torch.no_grad():
        out_tensor = generation(
            encoded_tensor,
            max_new_tokens=50,
            model=model,
            context_size=context_size,
            temperature=0,
            top_k=None
        )
    return tokenizer.decode(out_tensor.squeeze(0).tolist()).replace("\n", " ")


In [50]:
'''
我的 training loop 要管理哪些状态？

1. 会被训练改变的对象： model, optimizer
2. 不参与训练、只提供数据的对象：train_loader，eval_loader
3. 只用于评估的对象：val_data
4. 每个 batch 必须发生的动作：传数据，计算loss，清空梯度、反向传播计算新梯度，optimizer更新梯度
5. 每隔 eval_freq 才发生的动作：记录当前训练进度，在train和eval训练集上计算梯度并展示
6. 每个 epoch 结束才发生的动作：生成文本，看模型预测效果
7. 最后函数应该返回什么，为什么？返回train、val 的losses记录以便画线分析，tokens_seen用于横向对比等。
'''
def train(model, optimizer, train_loader, val_loader, eval_freq, num_epochs, start_text, tokenizer, eval_iter):
    train_losses = []
    val_losses=[]
    tokens_seen = 0 #用于真实衡量进度（训练了多少billion token比epoch、step更接近真实模型情况，而且便于横向对比）
    global_step = -1
    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            #计算损失、反向传播
            optimizer.zero_grad()
            loss = cal_loss_batch(input_batch, target_batch, model)
            loss.backward()
            optimizer.step()
            global_step += 1
            tokens_seen += input_batch.numel()
            if global_step % eval_freq == 0:
                model.eval()
                with torch.no_grad():
                    train_loss = cal_loss_loader(train_loader, model, eval_iter)
                    val_loss = cal_loss_loader(val_loader, model, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                print(f"Epoch{epoch+1} Step{global_step}",f"Train_loss:{train_loss}, Val_loss:{val_loss}")
                model.train()
        print(generate_text_sample(start_text,model,tokenizer))
    return train_losses, val_losses, tokens_seen

In [51]:
#训练
settings = {
    "learning_rate": 5e-4,
    "num_epochs":10,
    "batch_size":2,
    "weight_decay":0.1
}
torch.manual_seed(1980)
device = torch.device("cuda")
cfg = GPT_CONFIG_124M
file_path = "ch02/the-verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    data = file.read()
#创建训练和验证数据集
split_idx = int(0.9*len(data))
train_loader = create_dataloader(data[:split_idx], 
                                 batch_size=settings["batch_size"], 
                                 max_length=cfg["context_length"],
                                 stride=cfg["context_length"])
val_loader = create_dataloader(data[split_idx:],
                               batch_size=settings["batch_size"], 
                               max_length=cfg["context_length"],
                               stride=cfg["context_length"])
#初始化模型
model = GPTModel(cfg)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), 
                              lr=settings["learning_rate"], weight_decay=settings["weight_decay"])
tokenizer = tiktoken.get_encoding("gpt2")
train_losses, val_losses, tokens_seen = train(
    model, optimizer, train_loader, val_loader, eval_freq=5, eval_iter=1,
    num_epochs=settings["num_epochs"],
    start_text="Every effort moves you", tokenizer=tokenizer
)

Epoch1 Step0 Train_loss:9.91940689086914, Val_loss:9.819511413574219
Epoch1 Step5 Train_loss:8.048160552978516, Val_loss:7.999637603759766
Every effort moves you, the,, the,,, the,,, the the, the,, the, the, the,,, the,, the,,,,,,,, the,, the, the,,,,,,
Epoch2 Step10 Train_loss:6.530869483947754, Val_loss:6.78892183303833
Epoch2 Step15 Train_loss:5.8744707107543945, Val_loss:6.593161582946777
Every effort moves you, the the the the the the, the the, the the the the the the the the the the, the the, the the, the the, the the, the, the, the the, the the the the, the the, the
Epoch3 Step20 Train_loss:5.467774391174316, Val_loss:6.477010250091553
Epoch3 Step25 Train_loss:5.881923675537109, Val_loss:6.43940544128418
Every effort moves you.                                                 
Epoch4 Step30 Train_loss:5.663976192474365, Val_loss:6.521196365356445
Epoch4 Step35 Train_loss:5.288442611694336, Val_loss:6.495823383331299
Every effort moves you a a was. "--as it--as a on his pictures i

In [ ]:
torch.save({
"model_state_dict": model.state_dict(),
"optimizer_state_dict": optimizer.state_dict(),
},
"model_and_optimizer.pth"
)